In [ ]:

import pickle as pkl

import pandas as pd

In [ ]:
trainData = pd.read_csv("trainData.csv")  # 读取训练数据
testData = pd.read_csv("testData.csv")  # 读取测试数据
with open("batches.meta", 'rb') as f:
    labels = pkl.load(f)  # 读取标签数据
label_names = labels['label_names']  # 获取标签名称
trainData.shape, testData.shape, label_names

In [ ]:
# 将读取出来的数据转存为二进制文件，方便后续使用
with open("dataBatches", 'wb') as f:
    pkl.dump((trainData, testData, label_names), f)

In [ ]:
import matplotlib.pyplot as plt
import pickle as pkl

with open("dataBatches", 'rb') as f:
    trainData, testData, label_names = pkl.load(f)

In [ ]:
# 显示训练数据的图片
images = trainData.iloc[:, :-1]  # 获取训练数据的像素值
labels = trainData.iloc[:, -1]  # 获取训练数据的标签
categraies_num = len(label_names)  # 获取类别数量

for i, c in enumerate(label_names):
    image_list = images[labels == i].sample(7, replace=False)  # 从每个类别中随机选择7张图片，无放回
    for j, ind in enumerate(image_list.index):
        plt.subplot(7, categraies_num, j * categraies_num + i + 1)  # 设置子图位置
        # 先将单独的一张图片还原成3*32*32的原始形状
        # 再通过transpose函数将通道维度移到最后，修改为32*32*3的形状
        img = image_list.loc[ind].values.reshape(3, 32, 32).transpose(1, 2, 0)  # 将像素值转换为图像格式
        plt.imshow(img)  # 显示图像
        plt.axis("off")  # 关闭坐标轴显示

        if j == 0:
            plt.title(c, fontdict={'fontsize': 6})
plt.show()  # 显示所有图像

In [ ]:
# 重新采样n条数据集
def get_samples(data, n=200):
    # 按照标签分组，并从每个标签中随机采样n条数据，无放回
    result = data.groupby('label').sample(n, replace=False)
    return result


sample_data = get_samples(trainData, n=5000)  # 从训练数据中采样200条数据
sample_data.shape

In [ ]:

import numpy as np
from tqdm import tqdm
from PIL import Image
from torchvision import transforms
from sklearn.model_selection import train_test_split


def _build_normalize_params(normalize_params, is_grey):
    """
    内部工具函数：根据是否灰度图，给默认的 mean/std。
    """
    if normalize_params is not None:
        return normalize_params

    if is_grey:
        # 常用灰度图默认归一化参数
        return {'mean': [0.5], 'std': [0.5]}
    else:
        # ImageNet 风格默认参数
        return {
            'mean': [0.485, 0.456, 0.406],
            'std': [0.229, 0.224, 0.225]
        }


def _build_augmentation_params(augmentation_params, is_grey):
    """
    内部工具函数：根据是否灰度图，给默认的增强参数。
    """
    if augmentation_params is not None:
        return augmentation_params

    if is_grey:
        return {
            'horizontal_flip': True,
            'rotation': 10,
            'brightness': 0.2,
            'contrast': 0.2
        }
    else:
        return {
            'horizontal_flip': True,
            'rotation': 10,
            'brightness': 0.2,
            'contrast': 0.2,
            'saturation': 0.2,
            'hue': 0.2
        }


def preprocess_batch(flattened_arrays, labels, image_shape, resize_shape, normalize_params,
                     augmentation_params, augmentation_num, flatten, is_grey):
    """
     预处理一批展平的图片数据，包括可选的数据增强和归一化。

    参数:
        flattened_arrays (np.ndarray): 批量展平的输入图片数据，每一行代表一个展平的图片。
        labels (np.ndarray): 对应的标签数组。
        image_shape (tuple): 展平前的图片形状，格式为 (H, W, C) 或 (H, W)。
        resize_shape (tuple): 图像缩放的尺寸，格式为 (height, width)。
        normalize_params (dict): 归一化的均值和标准差参数，格式为 {'mean': [...], 'std': [...]}。
        augmentation_params (dict): 数据增强的参数。
        augmentation_num (int): 需要增强的样本数量。
        flatten (bool): 是否将图像展平成一维向量。
        is_grey (bool): 图像是否是灰度图。

    返回:
        tuple: (np.ndarray, np.ndarray) 预处理后的图像批量数组和对应的标签数组。
    """
    # 1. 恢复原始图像形状
    if image_shape is not None:
        images_np = flattened_arrays.reshape((-1,) + image_shape).astype(np.uint8)
    else:
        images_np = flattened_arrays.astype(np.uint8)

    # 2. 处理归一化和增强参数
    normalize_params = _build_normalize_params(normalize_params, is_grey)

    if augmentation_num > 0:
        augmentation_params = _build_augmentation_params(augmentation_params, is_grey)

    # 3. 构造基础预处理 transform（所有样本都会走）
    base_transform_ops = [
        transforms.Resize(resize_shape),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=normalize_params['mean'],
            std=normalize_params['std']
        )
    ]
    base_transform = transforms.Compose(base_transform_ops)

    # 4. 构造带增强的 transform（仅增强样本使用）
    aug_transform = None
    if augmentation_num > 0:
        aug_ops = []

        if augmentation_params.get('horizontal_flip', False):
            aug_ops.append(transforms.RandomHorizontalFlip())

        rotation_deg = augmentation_params.get('rotation', 0)
        if rotation_deg and rotation_deg != 0:
            aug_ops.append(transforms.RandomRotation(rotation_deg))

        # 颜色抖动（灰度图没有饱和度和色相）
        if is_grey:
            aug_ops.append(
                transforms.ColorJitter(
                    brightness=augmentation_params.get('brightness', 0),
                    contrast=augmentation_params.get('contrast', 0),
                )
            )
        else:
            aug_ops.append(
                transforms.ColorJitter(
                    brightness=augmentation_params.get('brightness', 0),
                    contrast=augmentation_params.get('contrast', 0),
                    saturation=augmentation_params.get('saturation', 0),
                    hue=augmentation_params.get('hue', 0),
                )
            )

        # 增强 + 预处理 一次完成
        aug_transform = transforms.Compose(aug_ops + base_transform_ops)

    # 5. NumPy -> PIL
    pil_mode = "L" if is_grey else "RGB"
    pil_images = [
        Image.fromarray(img, mode=pil_mode) for img in images_np
    ]

    # 6. 先处理原始样本
    processed_tensors = []
    for img in tqdm(pil_images, desc='Processing original images', unit='images'):
        tensor = base_transform(img)
        processed_tensors.append(tensor)

    labels_out = labels.copy()

    # 7. 再生成增强样本（如果需要）
    if augmentation_num > 0 and aug_transform is not None:
        idx_aug = np.random.choice(len(pil_images), augmentation_num, replace=True)
        aug_labels = labels[idx_aug]

        for idx in tqdm(idx_aug, desc='Augmenting images', unit='images'):
            img_aug = aug_transform(pil_images[idx])
            processed_tensors.append(img_aug)

        labels_out = np.concatenate([labels_out, aug_labels], axis=0)

    # 8. 堆叠成一个 batch：形状 [N, C, H, W]
    batch_tensor = torch.stack(processed_tensors, dim=0)

    # 9. 如果需要，将每张图 flatten 成一维向量
    if flatten:
        batch_tensor = batch_tensor.view(batch_tensor.size(0), -1)

    # 10. 返回 np.ndarray（保持你原来的函数签名）
    batch_array = batch_tensor.numpy()
    return batch_array, labels_out


def split_and_preprocess_data(flattened_arrays, labels, image_shape=None, test_size=0.2,
                              random_state=None, resize_shape=(224, 224), normalize_params=None,
                              augmentation_params=None, augmentation_num=0, flatten=False, is_grey=False):
    """
    分割数据集并预处理展平的图片数据。

    参数:
        flattened_arrays (np.ndarray): 批量展平的输入图片数据，每一行代表一个展平的图片。
        labels (np.ndarray): 对应的标签数组。
        test_size (float): 测试集比例，默认是0.2。
        random_state (int): 随机种子，确保数据分割的可重复性。
        image_shape (tuple): 展平前的图片形状。
        resize_shape (tuple): 图像缩放的尺寸，默认是 (224, 224)。
        normalize_params (dict): 归一化的均值和标准差参数。
        augmentation_params (dict): 数据增强的参数。
        augmentation_num (int): 需要增强的样本数量，默认为 0 表示不进行数据增强。
        flatten (bool): 是否将图像展平成一维向量。
        is_grey (bool): 图像是否是灰度图。

    返回:
        tuple: (np.ndarray, np.ndarray, np.ndarray, np.ndarray)
               分别为预处理后的 X_train, X_test, y_train, y_test。
    """

    # 1. 分割数据集
    X_train, X_test, y_train, y_test = train_test_split(
        flattened_arrays,
        labels,
        test_size=test_size,
        random_state=random_state
    )

    # 2. 预处理训练集（含可选增强）
    X_train_processed, y_train_processed = preprocess_batch(
        X_train,
        y_train,
        image_shape,
        resize_shape,
        normalize_params,
        augmentation_params,
        augmentation_num,
        flatten,
        is_grey=is_grey
    )

    # 3. 预处理测试集（不做增强）
    X_test_processed, y_test_processed = preprocess_batch(
        X_test,
        y_test,
        image_shape,
        resize_shape,
        normalize_params,
        augmentation_params=None,
        augmentation_num=0,
        flatten=flatten,
        is_grey=is_grey
    )

    return X_train_processed, X_test_processed, y_train_processed, y_test_processed

In [ ]:
X = sample_data.iloc[:, :-1].to_numpy()
y = sample_data.iloc[:, -1].to_numpy()

X_train, X_val, y_train, y_val = split_and_preprocess_data(X, y, image_shape=(32, 32, 3), resize_shape=(32, 32),
                                                           flatten=True)
X_train.shape, X_val.shape, y_train.shape, y_val.shape

In [ ]:
# 搭建全连接神经网络
import torch.nn as nn  # 导入神经网络模块
import torch.nn.functional as F  # 导入神经网络函数模块


class FCNet(nn.Module):
    def __init__(self):
        super().__init__()  # 调用父类的初始化方法
        self.fc1 = nn.Linear(3 * 32 * 32, 1024)  # 定义第一层全连接层，输入维度为3*32*32，输出维度为64
        self.fc2 = nn.Linear(1024, 512)  # 定义第二层全连接层，输入维度为64，输出维度为128
        self.fc3 = nn.Linear(512, 256)  # 定义第三层全连接层，输入维度为128，输出维度为10
        self.fc4 = nn.Linear(256, 128)  # 定义第四层全连接层，输入维度为128，输出维度为10
        self.fc5 = nn.Linear(128, 10)  # 定义第五层全连接层，输入维度为128，输出维度为10
        self.BN1 = nn.BatchNorm1d(1024)  # 定义第一层批归一化层，输入维度为64
        self.BN2 = nn.BatchNorm1d(512)  # 定义第二层批归一化层，输入维度为128

    def forward(self, x):
        x = self.fc1(x)  # 将输入数据通过第一层全连接层
        x = self.BN1(x)  # 对第一层的输出进行批归一化处理
        x = F.relu(x)  # 对第一层的输出进行ReLU激活函数处理
        x = F.dropout(x, p=0.3)  # 对第一层的输出进行Dropout处理，丢弃率为0.3
        x = self.fc2(x)  # 将第一层的输出通过第二层全连接层
        x = self.BN2(x)  # 对第二层的输出进行批归一化处理
        x = F.relu(x)  # 对第二层的输出进行ReLU激活函数处理
        x = F.dropout(x, p=0.3)  # 对第二层的输出进行Dropout处理，丢弃率为0.3
        x = F.relu(self.fc3(x))  # 将第二层的输出通过第三层全连接层
        x = F.relu(self.fc4(x))  # 将第三层的输出通过第四层全连接层
        x = self.fc5(x)  # 对第四层的输出进行Softmax函数处理，得到每个类别的概率分布
        return x  # 返回最终的输出结果


model1 = FCNet()
model1

In [ ]:
# 打包数据集
from torch.utils.data import TensorDataset, DataLoader  # 导入数据集和数据加载器模块
import torch

# 将数据转换为PyTorch的Tensor格式，并创建TensorDataset对象

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.long))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                            torch.tensor(y_val, dtype=torch.long))
# 创建DataLoader对象，设置批量大小和是否打乱数据
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
import torch.optim as optim  # 导入优化器模块

model1 = FCNet()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 检测是否有可用的GPU设备
model1.to(device)  # 将模型移动到指定设备上


def train(model, train_loader, lr=0.5, num_epoches=100):
    model.train()  # 设置模型为训练模式,训练模式会启用梯度计算和参数更新
    optimizer = optim.SGD(model.parameters(), lr=lr)  # 创建SGD优化器对象，传入模型的参数和学习率
    criterion = nn.CrossEntropyLoss()  # 定义交叉熵损失函数对象，用于计算模型输出和真实标签之间的损失值

    for epoch in range(num_epoches):  # 循环训练指定的轮数
        total_loss = 0  # 初始化总损失为0
        for X_batch, y_batch in train_loader:  # 遍历训练数据加载器中的每个批次数据
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)  # 将输入数据和标签移动到指定设备上
            optimizer.zero_grad()  # 清零优化器的梯度缓存
            outputs = model(X_batch)  # 将输入数据通过模型进行前向传播，得到输出结果
            loss = criterion(outputs, y_batch)  # 计算损失函数，使用one-hot编码将标签转换为与输出维度相同的格式
            loss.backward()  # 反向传播计算梯度
            optimizer.step()  # 更新模型参数
            total_loss += loss.item()  # 累加当前批次的损失值

        avg_loss = total_loss / len(train_loader)  # 计算平均损失值

        print(f"Epoch {epoch + 1}/{num_epoches}, Loss: {avg_loss:.4f}")  # 打印当前轮数和平均损失值


train(model1, train_loader)

In [ ]:
def evaluate(model, val_loader):
    model.eval()  # 设置模型为评估模式,评估模式会禁用梯度计算和参数更新
    correct = 0  # 初始化正确预测的样本数量为0
    total = 0  # 初始化总样本数量为0

    with torch.no_grad():  # 在评估过程中禁用梯度计算，以节省内存和计算资源
        for X_batch, y_batch in val_loader:  # 遍历验证数据加载器中的每个批次数据
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)  # 将输入数据和标签移动到指定设备上
            outputs = model(X_batch)  # 将输入数据通过模型进行前向传播，得到输出结果
            _, predicted = torch.max(outputs.data, 1)  # 获取输出结果中概率最大的类别作为预测结果
            total += y_batch.size(0)  # 累加当前批次的样本数量
            correct += (predicted == y_batch).sum().item()  # 累加当前批次中正确预测的样本数量

    accuracy = correct / total * 100  # 计算准确率
    print(f"Validation Accuracy: {accuracy:.2f}%")  # 打印验证准确率


evaluate(model1, val_loader)

In [ ]:
# x = sample_data.iloc[:, :-1].to_numpy()
# y = sample_data.iloc[:, -1].to_numpy()
# X_train, X_val, y_train, y_val = tr(x, y, test_size=0.2, random_state=42)
# X_train.shape, X_val.shape, y_train.shape, y_val.shape